In [8]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import MinMaxScaler

# Config
USER_ID = 1
INPUT_ANNOTATION = '/home/rounak/CODE/Low_Engagement_Detection/Data_preprocess/Preprocess_Signals/raw_data/Annotations/user1_annotations.csv'
INPUT_PHYSIO = '/home/rounak/CODE/Low_Engagement_Detection/Data_preprocess/Preprocess_Signals/raw_data/Physiological_signals/cleaned_user1_physiological.csv'
OUTPUT_DIR = 'scores/'
WINDOW_DURATION_MS = 5000  # 5-second window



In [9]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load annotation data
annotations = pd.read_csv(INPUT_ANNOTATION)
annotations.columns = ['time', 'valence', 'arousal', 'videoID']
annotations['time'] = annotations['time'].astype(np.int64)

# # Load physiological data (no header in original file)
# physio = pd.read_csv(INPUT_PHYSIO, header=None)
# physio.columns = ['ArduinoTime_ms', 'GSR', 'HR', 'SystemTime_ms', 'Datetime']
# physio['SystemTime_ms'] = physio['SystemTime_ms'].astype(np.int64)

# Step 1: Load with no headers
# physio = pd.read_csv(INPUT_PHYSIO, header=None)
physio = pd.read_csv(INPUT_PHYSIO, header=None, sep=',', engine='python')

physio.columns = ['ArduinoTime_ms', 'GSR', 'Pulse', 'SystemTime_ms', 'Datetime']

# Step 2: Remove row where column name slipped into data
physio = physio[physio['SystemTime_ms'] != 'SystemTime_ms']

# Step 3: Convert SystemTime_ms safely
physio['SystemTime_ms'] = pd.to_numeric(physio['SystemTime_ms'], errors='coerce')
physio = physio.dropna(subset=['SystemTime_ms'])  # Drop rows with NaNs
physio['SystemTime_ms'] = physio['SystemTime_ms'].astype(np.int64)

# Drop rows with invalid/missing values
# physio = physio.dropna(subset=['SystemTime_ms', 'GSR', 'HR'])
physio['SystemTime_ms'] = physio['SystemTime_ms'].astype(np.int64)


# Convert for filtering
physio['Datetime'] = pd.to_datetime(physio['SystemTime_ms'], unit='ms')


In [13]:
import pandas as pd
import numpy as np

# Load physiological data with headers already present
physio = pd.read_csv(INPUT_PHYSIO)

# Drop rows where header might be repeated as data
physio = physio[physio['SystemTime_ms'] != 'SystemTime_ms']

# Convert numeric columns safely
physio['GSR'] = pd.to_numeric(physio['GSR'], errors='coerce')
physio['Pulse'] = pd.to_numeric(physio['Pulse'], errors='coerce')
physio['SystemTime_ms'] = pd.to_numeric(physio['SystemTime_ms'], errors='coerce')

# Drop any rows with NaNs in critical columns
physio = physio.dropna(subset=['GSR', 'Pulse', 'SystemTime_ms'])

# Cast time column to integer
physio['SystemTime_ms'] = physio['SystemTime_ms'].astype(np.int64)

# Ensure datetime column is accurate
physio['Datetime'] = pd.to_datetime(physio['SystemTime_ms'], unit='ms')


In [10]:
# Find bad rows
bad_gsr_rows = physio[~physio['GSR'].astype(str).str.replace('.', '', regex=False).str.isnumeric()]
print("Bad GSR rows:", bad_gsr_rows.shape[0])
print(bad_gsr_rows.head(3))


Bad GSR rows: 0
Empty DataFrame
Columns: [ArduinoTime_ms, GSR, Pulse, SystemTime_ms, Datetime]
Index: []


In [11]:
physio.head()

,ArduinoTime_ms,GSR,Pulse,SystemTime_ms,Datetime
1,0,628,515,1751709909125,2025-07-05 10:05:09.125
2,86081228,628,516,1751709909225,2025-07-05 10:05:09.225
3,8600,628,515,1751709909325,2025-07-05 10:05:09.325
4,86081228,628,516,1751709909425,2025-07-05 10:05:09.425
5,8600,628,515,1751709909525,2025-07-05 10:05:09.525


In [14]:

for video_id in annotations['videoID'].unique():
    # Get time bounds for this video
    video_ann = annotations[annotations['videoID'] == video_id]
    start_time = video_ann['time'].min()
    end_time = video_ann['time'].max() + 10000  # Add buffer

    # Extract relevant signal segment
    segment = physio[(physio['SystemTime_ms'] >= start_time) & (physio['SystemTime_ms'] <= end_time)].copy()
    if segment.empty:
        print(f"[Skip] No signal data for video {video_id}")
        continue

    window_scores = []
    time_bounds = []

    current_start = segment['SystemTime_ms'].min()
    current_end = current_start + WINDOW_DURATION_MS

    while current_end <= segment['SystemTime_ms'].max():
        window = segment[(segment['SystemTime_ms'] >= current_start) & (segment['SystemTime_ms'] < current_end)]
        if len(window) > 1:
            gsr_mean = window['GSR'].mean()
            hr_mean = window['Pulse'].mean()
            time_bounds.append((current_start, (current_start + current_end) // 2, current_end))
            window_scores.append([gsr_mean, hr_mean])
        current_start = current_end
        current_end += WINDOW_DURATION_MS

    if len(window_scores) < 2:
        print(f"[Skip] Not enough valid windows for video {video_id}")
        continue

    # Compute difference between windows
    features = np.array(window_scores)
    diffs = np.linalg.norm(np.diff(features, axis=0), axis=1)
    diffs = np.insert(diffs, 0, diffs[0])  # pad first

    # Normalize scores
    norm_scores = MinMaxScaler().fit_transform(diffs.reshape(-1, 1)).flatten()

    # Create output dataframe
    result_rows = []
    for (start, border, end), score in zip(time_bounds, norm_scores):
        result_rows.append([start, border, end, score])

    df_scores = pd.DataFrame(result_rows, columns=['Start', 'Border', 'End', 'Score'])
    file_id = int(f"{USER_ID}{video_id}")
    df_scores.to_csv(f"{OUTPUT_DIR}{file_id}_scores.csv", index=False)
    print(f"[OK] Wrote: {OUTPUT_DIR}{file_id}_scores.csv")


[OK] Wrote: scores/10_scores.csv
[OK] Wrote: scores/15_scores.csv
[OK] Wrote: scores/13_scores.csv
[OK] Wrote: scores/17_scores.csv
[OK] Wrote: scores/18_scores.csv
[OK] Wrote: scores/11_scores.csv
[OK] Wrote: scores/12_scores.csv
[OK] Wrote: scores/14_scores.csv
[OK] Wrote: scores/16_scores.csv


In [15]:
df = pd.read_csv('/home/rounak/CODE/Low_Engagement_Detection/Data_preprocess/Preprocess_Signals/42_user_wise_normalized_features.csv')
print(df.columns)


Index(['Unnamed: 0', 'GSR_mean', 'GSR_variance', 'HR_mean', 'HR_variance',
       '75_percentile_GSR', '75_percentile_HR', 'GSRmean_persen_diff',
       'HRmean_persent_diff', 'GSRmean_diff', 'HRmean_diff',
       'valence_acc_video', 'arousal_acc_video', 'P_id', 'video_id', 'Score',
       'time', 'valence', 'arousal', 'videoID', 'start_time', 'end_time',
       'P_id.1', 'probe'],
      dtype='object')


In [2]:
import pandas as pd
import numpy as np
import os
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.preprocessing import MinMaxScaler

# ==== RuLSIF Core ====

def rulsif_score(X_ref, X_test, alpha=0.1, sigma=1.0):
    """
    Computes the RuLSIF-based change-point score (relative Pearson divergence).
    X_ref and X_test are (n_samples, n_features) numpy arrays.
    """
    n = X_ref.shape[0]
    Z = np.vstack([X_ref, X_test])  # combined for kernel centers

    # Compute kernel matrix
    K = rbf_kernel(Z, Z, gamma=1.0 / (2 * sigma ** 2))

    # Split into components
    H = alpha * K[:n, :n].dot(K[:n, :n].T) / n + (1 - alpha) * K[n:, :n].dot(K[n:, :n].T) / n
    h = K[:n, :n].mean(axis=0)

    # Solve for theta (ridge regression)
    reg = 1e-3
    theta = np.linalg.solve(H + reg * np.eye(n), h)

    # Estimate r_hat on both sets
    g_ref = K[:n, :n].dot(theta)
    g_test = K[n:, :n].dot(theta)

    # Relative Pearson divergence
    PE_alpha = -alpha / (2 * n) * np.sum(g_ref ** 2) \
               - (1 - alpha) / (2 * n) * np.sum(g_test ** 2) \
               + np.mean(g_ref) - 0.5
    return PE_alpha

# ==== Config ====
USER_ID = 1
INPUT_ANNOTATION = '/home/rounak/CODE/Low_Engagement_Detection/Data_preprocess/Preprocess_Signals/raw_data/Annotations/user1_annotations.csv'
INPUT_PHYSIO = '/home/rounak/CODE/Low_Engagement_Detection/Data_preprocess/Preprocess_Signals/raw_data/Physiological_signals/cleaned_user1_physiological.csv'
OUTPUT_DIR = 'scores/'
WINDOW_DURATION_MS = 5000  # 5 seconds
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==== Load and clean annotation ====
annotations = pd.read_csv(INPUT_ANNOTATION)
annotations.columns = ['time', 'valence', 'arousal', 'videoID']
annotations['time'] = annotations['time'].astype(np.int64)

# ==== Load and clean physiological data ====
physio = pd.read_csv(INPUT_PHYSIO, header=None, sep=',', engine='python')
physio.columns = ['ArduinoTime_ms', 'GSR', 'Pulse', 'SystemTime_ms', 'Datetime']
physio = physio[physio['SystemTime_ms'] != 'SystemTime_ms']
physio['SystemTime_ms'] = pd.to_numeric(physio['SystemTime_ms'], errors='coerce')
physio['GSR'] = pd.to_numeric(physio['GSR'], errors='coerce')
physio['Pulse'] = pd.to_numeric(physio['Pulse'], errors='coerce')
physio = physio.dropna(subset=['SystemTime_ms', 'GSR', 'Pulse'])
physio['SystemTime_ms'] = physio['SystemTime_ms'].astype(np.int64)

# ==== Process each video ====
for video_id in annotations['videoID'].unique():
    video_ann = annotations[annotations['videoID'] == video_id]
    start_time = video_ann['time'].min()
    end_time = video_ann['time'].max() + 10000

    segment = physio[(physio['SystemTime_ms'] >= start_time) & (physio['SystemTime_ms'] <= end_time)].copy()
    if segment.empty:
        print(f"[Skip] No signal data for video {video_id}")
        continue

    window_scores = []
    time_bounds = []

    current_start = segment['SystemTime_ms'].min()
    current_end = current_start + WINDOW_DURATION_MS

    windows = []
    bounds = []

    # Step 1: Create windows
    while current_end <= segment['SystemTime_ms'].max():
        window = segment[(segment['SystemTime_ms'] >= current_start) & (segment['SystemTime_ms'] < current_end)]
        if len(window) > 1:
            feature_vector = window[['GSR', 'Pulse']].values
            windows.append(feature_vector)
            bounds.append((current_start, (current_start + current_end) // 2, current_end))
        current_start = current_end
        current_end += WINDOW_DURATION_MS

    if len(windows) < 2:
        print(f"[Skip] Not enough windows for RuLSIF in video {video_id}")
        continue

    # Step 2: Compute RuLSIF score for each pair of adjacent windows
    scores = []
    for i in range(1, len(windows)):
        score = rulsif_score(windows[i-1], windows[i], alpha=0.1, sigma=1.0)
        scores.append(score)

    # Step 3: Pad first score to match bounds length
    scores = [scores[0]] + scores

    # Step 4: Normalize
    norm_scores = MinMaxScaler().fit_transform(np.array(scores).reshape(-1, 1)).flatten()

    # Step 5: Save results
    result_rows = []
    for (start, border, end), score in zip(bounds, norm_scores):
        result_rows.append([start, border, end, score])

    df_scores = pd.DataFrame(result_rows, columns=['Start', 'Border', 'End', 'Score'])
    file_id = int(f"{USER_ID}{video_id}")
    df_scores.to_csv(f"{OUTPUT_DIR}{file_id}_scores.csv", index=False)
    print(f"[✓] Wrote RuLSIF scores to: {OUTPUT_DIR}{file_id}_scores.csv")


[✓] Wrote RuLSIF scores to: scores/10_scores.csv
[✓] Wrote RuLSIF scores to: scores/15_scores.csv
[✓] Wrote RuLSIF scores to: scores/13_scores.csv
[✓] Wrote RuLSIF scores to: scores/17_scores.csv
[✓] Wrote RuLSIF scores to: scores/18_scores.csv
[✓] Wrote RuLSIF scores to: scores/11_scores.csv
[✓] Wrote RuLSIF scores to: scores/12_scores.csv
[✓] Wrote RuLSIF scores to: scores/14_scores.csv
[✓] Wrote RuLSIF scores to: scores/16_scores.csv


In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.preprocessing import MinMaxScaler

# ==== RuLSIF Core ====
def rulsif_score(X_ref, X_test, alpha=0.1, sigma=1.0):
    n = X_ref.shape[0]
    Z = np.vstack([X_ref, X_test])
    K = rbf_kernel(Z, Z, gamma=1.0 / (2 * sigma ** 2))
    H = alpha * K[:n, :n].dot(K[:n, :n].T) / n + (1 - alpha) * K[n:, :n].dot(K[n:, :n].T) / n
    h = K[:n, :n].mean(axis=0)
    reg = 1e-3
    theta = np.linalg.solve(H + reg * np.eye(n), h)
    g_ref = K[:n, :n].dot(theta)
    g_test = K[n:, :n].dot(theta)
    PE_alpha = -alpha / (2 * n) * np.sum(g_ref ** 2) \
               - (1 - alpha) / (2 * n) * np.sum(g_test ** 2) \
               + np.mean(g_ref) - 0.5
    return PE_alpha

# ==== Config ====
USER_ID = 1
INPUT_ANNOTATION = '/home/rounak/CODE/Low_Engagement_Detection/Data_preprocess/Preprocess_Signals/raw_data/Annotations/user1_annotations.csv'
INPUT_PHYSIO = '/home/rounak/CODE/Low_Engagement_Detection/Data_preprocess/Preprocess_Signals/raw_data/Physiological_signals/cleaned_user1_physiological.csv'
OUTPUT_DIR = 'scores/'
WINDOW_SIZE = 50  # fixed row-based window size
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==== Load annotation ====
annotations = pd.read_csv(INPUT_ANNOTATION)
annotations.columns = ['time', 'valence', 'arousal', 'videoID']
annotations['time'] = annotations['time'].astype(np.int64)

# ==== Load physiological signals ====
physio = pd.read_csv(INPUT_PHYSIO, header=None, sep=',', engine='python')
physio.columns = ['ArduinoTime_ms', 'GSR', 'Pulse', 'SystemTime_ms', 'Datetime']
physio = physio[physio['SystemTime_ms'] != 'SystemTime_ms']
physio['SystemTime_ms'] = pd.to_numeric(physio['SystemTime_ms'], errors='coerce')
physio['GSR'] = pd.to_numeric(physio['GSR'], errors='coerce')
physio['Pulse'] = pd.to_numeric(physio['Pulse'], errors='coerce')
physio = physio.dropna(subset=['SystemTime_ms', 'GSR', 'Pulse'])
physio['SystemTime_ms'] = physio['SystemTime_ms'].astype(np.int64)

# ==== Process each video ====
for video_id in annotations['videoID'].unique():
    video_ann = annotations[annotations['videoID'] == video_id]
    start_time = video_ann['time'].min()
    end_time = video_ann['time'].max() + 10000

    segment = physio[(physio['SystemTime_ms'] >= start_time) & (physio['SystemTime_ms'] <= end_time)].copy()
    if segment.empty:
        print(f"[Skip] No signal data for video {video_id}")
        continue

    feature_data = segment[['GSR', 'Pulse']].to_numpy()

    # Step 1: Create fixed-size row-based windows
    windows = []
    for i in range(0, len(feature_data) - WINDOW_SIZE + 1, WINDOW_SIZE):
        window = feature_data[i:i + WINDOW_SIZE, :]
        windows.append(window)

    if len(windows) < 2:
        print(f"[Skip] Not enough windows for RuLSIF in video {video_id}")
        continue

    # Step 2: Compute RuLSIF score for each pair of adjacent windows
    scores = []
    for i in range(1, len(windows)):
        score = rulsif_score(windows[i - 1], windows[i], alpha=0.1, sigma=1.0)
        scores.append(score)

    # Step 3: Pad first score
    scores = [scores[0]] + scores

    # Step 4: Normalize scores
    norm_scores = MinMaxScaler().fit_transform(np.array(scores).reshape(-1, 1)).flatten()

    # Step 5: Save scores
    df_scores = pd.DataFrame({'Score': norm_scores})
    file_id = int(f"{USER_ID}{video_id}")
    df_scores.to_csv(f"{OUTPUT_DIR}{file_id}_scores.csv", index=False)
    print(f"[✓] Wrote aligned RuLSIF scores to: {OUTPUT_DIR}{file_id}_scores.csv")


[✓] Wrote aligned RuLSIF scores to: scores/10_scores.csv
[✓] Wrote aligned RuLSIF scores to: scores/15_scores.csv
[✓] Wrote aligned RuLSIF scores to: scores/13_scores.csv
[✓] Wrote aligned RuLSIF scores to: scores/17_scores.csv
[✓] Wrote aligned RuLSIF scores to: scores/18_scores.csv
[✓] Wrote aligned RuLSIF scores to: scores/11_scores.csv
[✓] Wrote aligned RuLSIF scores to: scores/12_scores.csv
[✓] Wrote aligned RuLSIF scores to: scores/14_scores.csv
[✓] Wrote aligned RuLSIF scores to: scores/16_scores.csv


In [ ]:
kf=pd.read_csv()